<a href="https://colab.research.google.com/github/seantentansion-ux/Cleaned_Fruit_Vegetables_Farmgate_Prices/blob/main/ML_Model_and_evaluation_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import files
uploaded = files.upload()

Saving Cleaned_Palay_Inorganic_Fertilizer_Use_2003_2025.xlsx to Cleaned_Palay_Inorganic_Fertilizer_Use_2003_2025 (1).xlsx


In [9]:
import requests
import zipfile
import io

# URL of the dataset
url = "https://github.com/seantentansion-ux/Cleaned_Palay_Inorganic_Fertilizer_Use_2003_2025/blob/main/Cleaned_Palay_Inorganic_Fertilizer_Use_2003_2025.xlsx"


# Download the file
response = requests.get(url)
response.raise_for_status()

# Extract the zip file
with
zipfile.ZipFile(io.BytesIO(response.content))
as z:
    z.extractall("/content/")

print("GitHub dataset downloaded and extracted successfully.")

SyntaxError: expected ':' (325585698.py, line 14)

In [ ]:

# ============================================================
# PALAY INORGANIC FERTILIZER USE (2003-2025)
# DATA VISUALIZATION, ANALYTICS, AND MACHINE LEARNING
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score
)

# 1. LOAD DATASET
filename = next(iter(uploaded))
excel = pd.ExcelFile(filename)
print("Available sheets:", excel.sheet_names)

# Select the sheet containing regional/year data
sheet = (
    "By Region and Year"
    if "By Region and Year" in excel.sheet_names
    else excel.sheet_names[0]
)

df = pd.read_excel(filename, sheet_name=sheet)
df.columns = df.columns.str.strip()

print("\nDataset Preview:")
display(df.head())

print("\nDataset Information:")
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())

# 2. CLEAN DATA
df = df.drop_duplicates().copy()

# Convert numeric-looking columns into numbers
for col in df.columns:
    if col not in ["Region", "Year"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Display available columns to check the dataset structure
print("\nColumns:", df.columns.tolist())

# Use columns that exist in the actual dataset
if "All Grades" not in df.columns:
    raise ValueError(
        "The selected sheet has no 'All Grades' column. "
        "Check the printed sheet names and column names."
    )

# Fill missing numeric values with their medians
for col in df.select_dtypes(include=np.number).columns:
    df[col] = df[col].fillna(df[col].median())

# Fill missing categorical values
for col in df.select_dtypes(exclude=np.number).columns:
    df[col] = df[col].fillna("Unknown")

# Remove rows where target is still missing
df = df.dropna(subset=["All Grades"]).copy()

# 3. DATA VISUALIZATION

# A. Dataset size
print("\nNumber of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

# B. Line chart: average All Grades by year
if "Year" in df.columns:
    yearly = df.groupby("Year")["All Grades"].mean()

    plt.figure(figsize=(10, 5))
    yearly.plot(marker="o")
    plt.title("Average All Grades by Year")
    plt.xlabel("Year")
    plt.ylabel("Average All Grades")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# C. Bar chart: average All Grades by region
if "Region" in df.columns:
    regional = (
        df.groupby("Region")["All Grades"]
        .mean().sort_values(ascending=False).head(10)
    )

    plt.figure(figsize=(10, 5))
    regional.plot(kind="bar")
    plt.title("Top 10 Regions by Average All Grades")
    plt.xlabel("Region")
    plt.ylabel("Average All Grades")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

# D. Histogram
plt.figure(figsize=(8, 5))
plt.hist(df["All Grades"], bins=10, edgecolor="black")
plt.title("Distribution of All Grades")
plt.xlabel("All Grades")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

# E. Pie chart: share of records by region
if "Region" in df.columns:
    df["Region"].value_counts().plot(
        kind="pie", autopct="%1.1f%%", figsize=(8, 8)
    )
    plt.title("Percentage of Records by Region")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()

# 4. DESCRIPTIVE ANALYTICS
print("\nDescriptive Statistics:")
display(df.describe(include="all").T)

print("\nAll Grades Summary:")
print("Mean:", df["All Grades"].mean())
print("Median:", df["All Grades"].median())
print("Minimum:", df["All Grades"].min())
print("Maximum:", df["All Grades"].max())
print("Standard Deviation:", df["All Grades"].std())

# Correlation heatmap
numeric_df = df.select_dtypes(include=np.number)

if numeric_df.shape[1] >= 2:
    plt.figure(figsize=(10, 7))
    sns.heatmap(
        numeric_df.corr(),
        annot=True,
        cmap="coolwarm",
        fmt=".2f"
    )
    plt.title("Correlation Heatmap")
    plt.tight_layout()
    plt.show()

# 5. PREPARE CLASSIFICATION TARGET
# High = All Grades at or above median
# Low = All Grades below median

median_value = df["All Grades"].median()

df["Grade_Level"] = np.where(
    df["All Grades"] >= median_value, 1, 0
)

# Exclude All Grades to avoid directly using the target
# to predict itself. Exclude other total/derived target columns too.
exclude = ["All Grades", "Grade_Level"]

# Use component columns, year, and region as predictors
X = df.drop(columns=exclude, errors="ignore")
y = df["Grade_Level"]

# Drop columns that contain no usable information
X = X.dropna(axis=1, how="all")
X = X.loc[:, X.nunique(dropna=False) > 1]

# Drop columns that directly duplicate the target
# through a perfect numeric relationship, if any are found
# (Inspect your columns and interpret the dataset carefully.)

# Separate numeric and categorical columns
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(
    exclude=np.number
).columns.tolist()

# Preprocessing pipeline
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Target definition: 0 = Low, 1 = High")
print("Median threshold:", median_value)

# 6. FOUR CLASSIFICATION MODELS
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5, random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, random_state=42
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    )
}

results = []
predictions = {}
probabilities = {}
trained_models = {}

for name, algorithm in models.items():

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", algorithm)
    ])

    model.fit(X_train, y_train)

    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]

    trained_models[name] = model
    predictions[name] = pred
    probabilities[name] = prob

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(
            y_test, pred, zero_division=0
        ),
        "Recall": recall_score(
            y_test, pred, zero_division=0
        ),
        "F1-Score": f1_score(
            y_test, pred, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(y_test, prob)
    })

# 7. CONFUSION MATRICES
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, (name, pred) in zip(axes.flat, predictions.items()):
    cm = confusion_matrix(y_test, pred, labels=[0, 1])

    display_cm = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Low", "High"]
    )

    display_cm.plot(ax=ax, colorbar=False)
    ax.set_title(name)

plt.tight_layout()
plt.show()

# 8. ROC CURVES
plt.figure(figsize=(9, 7))

for name, prob in probabilities.items():
    fpr, tpr, thresholds = roc_curve(y_test, prob)
    auc_score = roc_auc_score(y_test, prob)

    plt.plot(
        fpr, tpr,
        label=f"{name} (AUC = {auc_score:.3f})"
    )

plt.plot(
    [0, 1], [0, 1],
    linestyle="--", label="Random Classifier"
)

plt.title("ROC Curve Comparison")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 9. MODEL COMPARISON TABLE
comparison = pd.DataFrame(results)
comparison = comparison.sort_values(
    by="F1-Score", ascending=False
).reset_index(drop=True)

print("\nMODEL COMPARISON TABLE")
display(comparison.round(4))

# Identify model with highest F1-score
best_model = comparison.iloc[0]

print("\nBest model by F1-Score:", best_model["Model"])
print("F1-Score:", round(best_model["F1-Score"], 4))
print("Accuracy:", round(best_model["Accuracy"], 4))
print("ROC-AUC:", round(best_model["ROC-AUC"], 4))

# 10. SAVE COMPARISON TABLE
comparison.to_csv("model_comparison_results.csv", index=False)

print("\nCompleted! Downloading comparison table...")

from google.colab import files
files.download("model_comparison_results.csv")



# CHECK ACCURACY AND DATASET SIZE

print("Original dataset records:", len(df))
print("Original dataset columns:", len(df.columns))

accuracy_threshold = 0.80

low_accuracy_models = comparison[
    comparison["Accuracy"] < accuracy_threshold
]

if not low_accuracy_models.empty:
    print("\nWARNING: Some models have accuracy below 80%.")
    display(low_accuracy_models[["Model", "Accuracy"]])

    print("""
NEXT STEPS:
1. Collect more genuine records from reliable sources.
2. Check missing values and incorrect data.
3. Review useful predictor columns.
4. Tune model parameters using cross-validation.
5. Re-evaluate using a separate test dataset.

Do not duplicate rows or add fabricated values just to
make the accuracy reach 80%.
""")
else:
    print("\nAll four models achieved at least 80% accuracy.")

print("\nCurrent dataset record count:", len(df))
